# Chunk-level Hybrid Retrieval (BM25 + E5 base with RRF)

Notebook này triển khai hệ thống **Hybrid Retrieval** kết hợp giữa:
1. **Dense Retrieval (E5 Base)**: Tìm kiếm ngữ nghĩa sử dụng mô hình `intfloat/multilingual-e5-base` và FAISS Index trên cấp độ Chunk (1.08 triệu đoạn).
2. **Sparse Retrieval (BM25)**: Tìm kiếm từ khóa sử dụng thuật toán `BM25Okapi` trên cấp độ Chunk.

Kết quả từ 2 phương pháp được kết hợp bằng thuật toán **Reciprocal Rank Fusion (RRF)** ở cấp độ Chunk. Sau đó, ta tiến hành gộp nhóm theo bài viết (`doc_id`) để trả về các bài báo tương đồng nhất.

## 1. Import thư viện và Cấu hình

In [1]:
import os
import gc
import time
import pickle
import faiss
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from pathlib import Path
from transformers import AutoTokenizer, AutoModel
from rank_bm25 import BM25Okapi

# Định nghĩa các đường dẫn
CHUNKS_PATH = Path("data/corpus_chunks.json")
CHUNK_DF_PATH = Path("data/news_dataset_chunked.pkl")
E5_DIR = Path("data/e5_base")
FAISS_INDEX_PATH = E5_DIR / "semantic_e5_base.faiss"
METADATA_PATH = E5_DIR / "chunks_metadata.parquet"
CONFIG_PATH = E5_DIR / "embedding_config.json"
BM25_MODEL_PATH = Path("indexes/bm25/bm25_model_chunks.pkl")
E5_MODEL_OFFLINE = Path("models/e5_base_offline")

print("Đã thiết lập cấu hình đường dẫn.")

d:\anaconda3\envs\it\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đã thiết lập cấu hình đường dẫn.


## 2. Load Mô hình E5 (Tìm kiếm ngữ nghĩa)

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print("Device đang sử dụng:", device)

# Load tokenizer và model offline
tokenizer = AutoTokenizer.from_pretrained(E5_MODEL_OFFLINE)
model = AutoModel.from_pretrained(
    E5_MODEL_OFFLINE,
    torch_dtype=dtype,
).to(device)
model.eval()

def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state.float() * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

@torch.no_grad()
def encode_texts(texts, batch_size=16, max_length=512):
    all_embeddings = []
    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        inputs = tokenizer(
            batch_texts,
            max_length=max_length,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            outputs = model(**inputs)

        embeddings = mean_pooling(
            outputs.last_hidden_state,
            inputs["attention_mask"],
        )
        embeddings = F.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu().numpy().astype("float32"))

        del inputs, outputs, embeddings
        if device == "cuda":
            torch.cuda.empty_cache()

    return np.vstack(all_embeddings)

print("Mô hình E5 base đã sẵn sàng.")

Device đang sử dụng: cuda


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3494.01it/s]


Mô hình E5 base đã sẵn sàng.


## 3. Nạp FAISS Index và Metadata

In [3]:
print("Đang nạp FAISS index...")
faiss_index = faiss.read_index(str(FAISS_INDEX_PATH))

print("Đang nạp chunks metadata...")
metadata = pd.read_parquet(METADATA_PATH)

print(f"-> Tổng số vector trong FAISS: {faiss_index.ntotal}")
print(f"-> Tổng số dòng metadata: {len(metadata)}")

Đang nạp FAISS index...
Đang nạp chunks metadata...
-> Tổng số vector trong FAISS: 1078604
-> Tổng số dòng metadata: 1078604


## 4. Xây dựng hoặc Nạp mô hình BM25 ở cấp độ Chunk

In [4]:
if BM25_MODEL_PATH.exists():
    print(f"Tìm thấy mô hình BM25 đã lưu tại: {BM25_MODEL_PATH}")
    print("Đang tiến hành load...")
    t0 = time.time()
    with open(BM25_MODEL_PATH, "rb") as f:
        bm25_model = pickle.load(f)
    print(f"-> Load thành công trong {time.time() - t0:.2f} giây.")
else:
    print("Không tìm thấy mô hình BM25 chunk đã lưu. Tiến hành xây dựng mới từ news_dataset_chunked.pkl...")
    t0 = time.time()
    df_chunks = pd.read_pickle(CHUNK_DF_PATH)
    print(f"-> Load DataFrame chunks mất {time.time() - t0:.2f} giây. Shape: {df_chunks.shape}")
    
    corpus_processed = df_chunks["chunk_processed"].fillna("").astype(str).tolist()
    del df_chunks
    gc.collect()
    
    print("-> Đang tách từ cho corpus (split)...")
    t0 = time.time()
    tokenized_corpus = [doc.split() for doc in corpus_processed]
    print(f"-> Tách từ hoàn tất trong {time.time() - t0:.2f} giây.")
    
    del corpus_processed
    gc.collect()
    
    print("-> Đang khởi tạo mô hình BM25Okapi...")
    t0 = time.time()
    bm25_model = BM25Okapi(tokenized_corpus)
    print(f"-> Xây dựng BM25 hoàn tất trong {time.time() - t0:.2f} giây.")
    
    del tokenized_corpus
    gc.collect()
    
    # Lưu mô hình xuống đĩa để tái sử dụng
    os.makedirs(BM25_MODEL_PATH.parent, exist_ok=True)
    print(f"-> Đang lưu mô hình xuống: {BM25_MODEL_PATH}...")
    with open(BM25_MODEL_PATH, "wb") as f:
        pickle.dump(bm25_model, f)
    print("-> Lưu thành công!")

Tìm thấy mô hình BM25 đã lưu tại: indexes\bm25\bm25_model_chunks.pkl
Đang tiến hành load...
-> Load thành công trong 24.50 giây.


## 5. Xây dựng các hàm Tìm kiếm Độc lập (BM25 & E5)

In [5]:
def search_bm25_chunks(query, top_k=100):
    """
    Tìm kiếm từ khóa bằng thuật toán BM25 trên danh sách các chunk.
    """
    tokenized_query = str(query).lower().split()
    scores = bm25_model.get_scores(tokenized_query)
    
    # Lấy top_k chỉ mục có điểm cao nhất
    top_k_idx = np.argsort(scores)[::-1][:top_k]
    
    results = []
    for idx in top_k_idx:
        score = scores[idx]
        if score <= 0:
            continue
        row = metadata.iloc[int(idx)].to_dict()
        row["bm25_score"] = float(score)
        row["chunk_index"] = int(idx)
        results.append(row)
        
    return pd.DataFrame(results)

def search_e5_chunks(query, top_k=100):
    """
    Tìm kiếm ngữ nghĩa bằng mô hình E5 và FAISS trên danh sách các chunk.
    """
    # Theo đặc tả của E5, query cần thêm prefix 'query: '
    query_text = "query: " + str(query).strip()
    query_embedding = encode_texts([query_text], batch_size=1)
    
    scores, ids = faiss_index.search(query_embedding, top_k)
    
    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue
        row = metadata.iloc[int(idx)].to_dict()
        row["e5_score"] = float(score)
        row["chunk_index"] = int(idx)
        results.append(row)
        
    return pd.DataFrame(results)

print("Các hàm tìm kiếm độc lập đã sẵn sàng.")

Các hàm tìm kiếm độc lập đã sẵn sàng.


## 6. Xây dựng thuật toán Hybrid Retrieval với RRF

In [6]:
def hybrid_retrieval_rrf(query, top_k=10, candidate_k=100, rrf_k=60):
    """
    Tìm kiếm Hybrid kết hợp BM25 và E5 ở cấp độ Chunk sử dụng Reciprocal Rank Fusion (RRF).
    """
    if not query or not str(query).strip():
        return pd.DataFrame()
        
    # 1. Chạy tìm kiếm thô với cả 2 retriever để lấy tập ứng viên
    df_bm25 = search_bm25_chunks(query, top_k=candidate_k)
    df_e5 = search_e5_chunks(query, top_k=candidate_k)
    
    # 2. Áp dụng RRF
    rrf_scores = {}
    chunk_data = {}
    
    # Duyệt qua danh sách BM25 và tính điểm
    if not df_bm25.empty:
        for rank, row in enumerate(df_bm25.itertuples(), 1):
            c_idx = row.chunk_index
            rrf_scores[c_idx] = rrf_scores.get(c_idx, 0.0) + 1.0 / (rrf_k + rank)
            if c_idx not in chunk_data:
                chunk_data[c_idx] = {
                    "doc_id": row.doc_id,
                    "chunk_id": row.chunk_id,
                    "url": row.url,
                    "topic": row.topic,
                    "text": row.text,
                    "bm25_score": row.bm25_score,
                    "e5_score": 0.0
                }
            else:
                chunk_data[c_idx]["bm25_score"] = row.bm25_score
                
    # Duyệt qua danh sách E5 và tính điểm
    if not df_e5.empty:
        for rank, row in enumerate(df_e5.itertuples(), 1):
            c_idx = row.chunk_index
            rrf_scores[c_idx] = rrf_scores.get(c_idx, 0.0) + 1.0 / (rrf_k + rank)
            if c_idx not in chunk_data:
                chunk_data[c_idx] = {
                    "doc_id": row.doc_id,
                    "chunk_id": row.chunk_id,
                    "url": row.url,
                    "topic": row.topic,
                    "text": row.text,
                    "bm25_score": 0.0,
                    "e5_score": row.e5_score
                }
            else:
                chunk_data[c_idx]["e5_score"] = row.e5_score
                
    # 3. Gom kết quả và sắp xếp theo điểm RRF
    results = []
    for c_idx, score in rrf_scores.items():
        item = chunk_data[c_idx].copy()
        item["rrf_score"] = score
        item["chunk_index"] = c_idx
        results.append(item)
        
    df_hybrid = pd.DataFrame(results)
    if not df_hybrid.empty:
        df_hybrid = df_hybrid.sort_values("rrf_score", ascending=False).reset_index(drop=True)
        return df_hybrid.head(top_k)
        
    return df_hybrid

print("Hàm Hybrid Retrieval với RRF đã cấu hình thành công.")

Hàm Hybrid Retrieval với RRF đã cấu hình thành công.


## 7. Hàm gộp kết quả theo bài viết (Document Grouping)

In [7]:
def group_hybrid_results_by_doc(df_hybrid, top_k=5):
    """
    Gộp các chunk thuộc cùng một bài báo (doc_id), giữ lại chunk có điểm RRF cao nhất đại diện.
    """
    if df_hybrid.empty:
        return df_hybrid
        
    grouped = (
        df_hybrid.sort_values("rrf_score", ascending=False)
        .groupby("doc_id", as_index=False)
        .first()
        .sort_values("rrf_score", ascending=False)
        .head(top_k)
    )
    return grouped.reset_index(drop=True)

print("Hàm group_hybrid_results_by_doc đã sẵn sàng.")

Hàm group_hybrid_results_by_doc đã sẵn sàng.


## 8. Trực quan hóa kết quả tìm kiếm

In [8]:
def display_search_results(df_results, title="Search Results"):
    print(f"\n{'='*40} {title.upper()} (Tìm thấy: {len(df_results)}) {'='*40}")
    if df_results.empty:
        print("Không tìm thấy kết quả nào tương đồng.")
        return
        
    for idx, row in df_results.iterrows():
        print(f"\n[Hạng {idx+1}] | Bài viết ID: {row.get('doc_id')} | Chunk ID: {row.get('chunk_id')}")
        
        if "rrf_score" in row:
            print(f"Điểm RRF: {row['rrf_score']:.6f} | Điểm BM25 thô: {row['bm25_score']:.2f} | Điểm E5 (Cosine): {row['e5_score']:.4f}")
        elif "bm25_score" in row:
            print(f"Điểm BM25 thô: {row['bm25_score']:.2f}")
        elif "e5_score" in row:
            print(f"Điểm E5 (Cosine): {row['e5_score']:.4f}")
            
        print(f"Chủ đề: {row.get('topic')} | URL: {row.get('url')}")
        print(f"Đoạn văn tiêu biểu: {row.get('text', '')[:350]}...")
        print("-" * 100)

## 9. Chạy Thử nghiệm và So sánh kết quả

In [9]:
query = "cướp tiệm vàng thừa thiên huế bằng súng ak"
print(f"Query thử nghiệm: '{query}'")

# 1. Tìm kiếm BM25
df_bm25 = search_bm25_chunks(query, top_k=5)
display_search_results(df_bm25, title="BM25 Search Only")

# 2. Tìm kiếm E5 semantic
df_e5 = search_e5_chunks(query, top_k=5)
display_search_results(df_e5, title="E5 Semantic Search Only")

# 3. Tìm kiếm Hybrid + RRF
df_hybrid = hybrid_retrieval_rrf(query, top_k=15, candidate_k=50)
df_hybrid_grouped = group_hybrid_results_by_doc(df_hybrid, top_k=5)
display_search_results(df_hybrid_grouped, title="Hybrid RRF Search (Gộp theo bài báo)")

Query thử nghiệm: 'cướp tiệm vàng thừa thiên huế bằng súng ak'

======================================== BM25 SEARCH ONLY (Tìm thấy: 5) ========================================

[Hạng 1] | Bài viết ID: 1709 | Chunk ID: 1709_2
Điểm BM25 thô: 56.47
Chủ đề: Pháp luật | URL: https://zingnews.vn/ke-mang-ak-cuop-tiem-vang-o-hue-doi-gap-pho-giam-doc-cong-an-tinh-post1340992.html
Đoạn văn tiêu biểu: Kẻ mang AK cướp tiệm vàng ở Huế đòi gặp phó giám đốc công an tỉnh. Theo lãnh đạo Công an tỉnh Thừa Thiên - Huế, nghi phạm sử dụng súng AK khi cướp tài sản. Một nguồn tin cho biết người cướp tiệm vàng ở TP Huế là Ngô Văn Quốc (38 tuổi, công tác tại Trại giam Bình Điền)....
----------------------------------------------------------------------------------------------------

[Hạng 2] | Bài viết ID: 1412 | Chunk ID: 1412_0
Điểm BM25 thô: 53.64
Chủ đề: Tin nhanh 24h | URL: https://vtc.vn/da-bat-duoc-nghi-pham-dung-sung-cuop-tiem-vang-tai-cho-dong-ba-ar691279.html
Đoạn văn tiêu biểu: Thừa Thiên - Huế: Bắ